In [1]:
config_path = "config/params.yaml"
mode = "default"

In [2]:
# Parameters
config_path = "config/params.yaml"


# 08 — Replikasi pilot pada jam dev

**Tujuan:** membuktikan pipeline mereproduksi angka pilot pada jam dev, sebelum parameter dikunci.

In [3]:
import numpy as np
import pandas as pl_pandas
import polars as pl

from ghremoval.config import load_context
from ghremoval.data import apply_age_bins, fit_age_edges, xy
from ghremoval.data import load_frame
from ghremoval.evaluate import ap_by_stratum
from ghremoval.models.pilot import oof_scores
from ghremoval.quality import Checks
from ghremoval.report import save_table
from ghremoval.runlog import RunLog
from ghremoval.splits import id_block_groups, stratified_group_folds

ctx = load_context(config_path)
run = RunLog("08_dev_hour_replication", ctx.paths.run_log, ctx.paths.root)
checks = Checks()
seeds = ctx.p.locked["splits"]["cv_seeds"]

## 1. Jumlah akun

In [4]:
frame = load_frame(ctx, 60, ["dev"]).filter(pl.col("in_fixed_cohort"))
frame = apply_age_bins(frame, fit_age_edges(frame["id_recency"].to_numpy()))
y = frame["is_gone"].to_numpy().astype(int)
strata = frame["stratum"].to_numpy()
ids = frame["actor_id"].to_numpy()
counts = {"Akun >=5": (frame.height, 8434), "5-19": (int((strata == "5-19").sum()), 7576), "20+": (int((strata == "20+").sum()), 858)}
gone = {"Hilang >=5": (int(y.sum()), 519), "Hilang 5-19": (int(y[strata == "5-19"].sum()), 185), "Hilang 20+": (int(y[strata == "20+"].sum()), 334)}
for name, (got, want) in counts.items():
    checks.add(name, got == want, f"{got} vs {want}")
for name, (got, want) in gone.items():
    checks.add(name, abs(got - want) <= 0.02 * want, f"{got} vs {want}")

[LULUS] Akun >=5: 8434 vs 8434
[LULUS] 5-19: 7576 vs 7576
[LULUS] 20+: 858 vs 858
[LULUS] Hilang >=5: 519 vs 519
[LULUS] Hilang 5-19: 185 vs 185
[LULUS] Hilang 20+: 334 vs 334


## 2. Model dengan konfigurasi pilot

In [5]:
PILOT = {("B2", "random"): (0.690, 0.090, 0.918), ("B2", "blocks_w20000"): (0.693, 0.084, 0.934), ("M_behav", "random"): (0.767, 0.251, 0.979), ("M_behav", "blocks_w20000"): (0.753, 0.197, 0.979), ("M_behav", "blocks_w100000"): (0.749, 0.196, 0.975)}
rows = []
for (feature_set, split), expected in PILOT.items():
    groups = np.arange(frame.height) if split == "random" else id_block_groups(ids, int(split.split("w")[1]))
    X, _ = xy(frame, feature_set)
    score = oof_scores(X, y, {seed: stratified_group_folds(groups, y, 5, seed) for seed in seeds})
    ap = ap_by_stratum(y, score, strata)
    got = (ap["all"], ap["5-19"], ap["20+"])
    rows.append({"features": feature_set, "split": split, "pilot_all": expected[0], "pipeline_all": got[0], "pilot_5_19": expected[1], "pipeline_5_19": got[1], "pilot_20p": expected[2], "pipeline_20p": got[2], "within_tolerance": all(abs(g - e) <= 0.03 for g, e in zip(got, expected))})
table = pl_pandas.DataFrame(rows)
save_table(table, "tab_dev_replication", "Development-hour replication of the pilot")
table

,features,split,pilot_all,pipeline_all,pilot_5_19,pipeline_5_19,pilot_20p,pipeline_20p,within_tolerance
0,B2,random,0.690,0.693519,0.090,0.083738,0.918,0.931183,True
1,B2,blocks_w20000,0.693,0.693086,0.084,0.084069,0.934,0.934285,True
2,M_behav,random,0.767,0.767035,0.251,0.244775,0.979,0.979657,True
3,M_behav,blocks_w20000,0.753,0.753659,0.197,0.198445,0.979,0.979069,True
4,M_behav,blocks_w100000,0.749,0.749554,0.196,0.194356,0.975,0.975577,True


## Ringkasan & cek

In [6]:
checks.add("Q14", bool(table["within_tolerance"].all()), f"{int(table['within_tolerance'].sum())}/{len(table)} baris dalam toleransi")
checks.raise_if_failed()
print(run.finish(n_rows=frame.height, n_gone=int(y.sum())))

[LULUS] Q14: 5/5 baris dalam toleransi
{'notebook': '08_dev_hour_replication', 'finished_at_utc': '2026-11-10T11:31:48+00:00', 'git_commit': 'abfcd4a', 'duration_seconds': 20.0, 'counts': {'n_rows': 8434, 'n_gone': 519}}
